# Sample NACC data for training

Created By: Sahana Kowshik

Date: 08/14/2025

In [1]:
import pandas as pd
import ast
import re
import random
import string
import warnings
warnings.filterwarnings('ignore')


# Load data

In [2]:
random_state = 42

In [3]:
# group_size = "gs_8"
group_size = "gs_16"

In [4]:
all_data = pd.read_csv("../../data/nacc/training_data/training_data_grpo_modified_cog_np_HASH/train_summary.csv")
print("Length of all_data:", len(all_data))

# filtered_data = pd.read_csv(f"../../data/nacc/training_data/training_data_grpo_modified_cog_np/skywork_style/{group_size}/train_filtered.csv")
# print("Length of filtered_data:", len(filtered_data))

# stage_wise_data = pd.read_csv(f"../../data/nacc/training_data/training_data_grpo_modified_cog_np/stage_wise/{group_size}/stage_wise.csv")
# print("Length of stage_wise_data:", len(stage_wise_data))

Length of all_data: 43064


# Methods

In [5]:
def add_cog_type(row):
    if "NC" in row['options'] and "MCI" in row['options']:
        row["COG_TYPE"] = "NC_MCI"
    elif "MCI" in row['options'] and "DE" in row['options']:
        row["COG_TYPE"] = "MCI_DE"
    elif "NC" in row['options'] and "DE" in row['options']:
        row["COG_TYPE"] = "NC_DE"
    return row

In [ ]:
def sample_cases(df, n=500, n_etpr=300):
    np_data = df[df['Q_TYPE'] == 'Neuropath']
    datscan_data = df[df['Q_TYPE'] == 'DATSCAN']
    amylpet_data = (
        df[df['Q_TYPE'] == "Amyloid PET"]
        .groupby('ground_truth_text', group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), n), random_state=random_state))  # Safe sampling
        .reset_index(drop=True)
    )
    amylcsf_data = (
        df[df['Q_TYPE'] == "Amyloid CSF"]
        .groupby('ground_truth_text', group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), n), random_state=random_state))  # Safe sampling
        .reset_index(drop=True)
    )
    etpr_data = (
        df[df['Q_TYPE'] == "Primary etiology"]
        .groupby('ground_truth_text', group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), n_etpr), random_state=random_state))  # Safe sampling
        .reset_index(drop=True)
    )
    cog_data = (
        df[df['Q_TYPE'] == "Cognitive status"].apply(add_cog_type, axis=1)
        .groupby('COG_TYPE', group_keys=False)
        .apply(
            lambda cog_grp: cog_grp
                .groupby('ground_truth_text', group_keys=False)
                .apply(lambda x: x.sample(n=min(len(x), n), random_state=random_state))
                # .sample(n=min(len(cog_grp), 2 * n), random_state=random_state)
                .reset_index(drop=True)
        )
        .reset_index(drop=True)
    )
    cog_data.drop(['COG_TYPE'], inplace=True, axis=1)
    
    # cog_data = change_cog_cases(df[df['Q_TYPE'] == 'Cognitive status'], df[df['Q_TYPE'] == 'MCI subtype'], n=n)
    combined_data = pd.concat([np_data, datscan_data, amylpet_data, amylcsf_data, etpr_data, cog_data]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    assert combined_data['ID'].is_unique, "combined_data has repeated IDs"
    
    return combined_data


In [ ]:
def sample_cases_oversample(df, n=500, n_etpr=300):
    # Neuropath
    np_df = df[df['Q_TYPE'] == "Neuropath"]
    group_sizes = np_df.groupby('ground_truth_text').size()
    max_n = group_sizes.max()
    def oversample_group(x):
        # Always include all original cases, then sample (with replacement) the remainder needed
        n_to_add = max_n - len(x)
        if n_to_add <= 0:
            return x
        else:
            return pd.concat([x, x.sample(n=n_to_add, replace=True, random_state=random_state)], ignore_index=True)
    np_data = (
        np_df
        .groupby('ground_truth_text', group_keys=False)
        .apply(oversample_group)
        .reset_index(drop=True)
    )
    
    # Datscan
    datscan_data = df[df['Q_TYPE'] == 'DATSCAN']
    
    # Amyloid
    amylpet_data = (
        df[df['Q_TYPE'] == "Amyloid PET"]
        .groupby('ground_truth_text', group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), n), random_state=random_state))  # Safe sampling
        .reset_index(drop=True)
    )
    amylcsf_data = (
        df[df['Q_TYPE'] == "Amyloid CSF"]
        .groupby('ground_truth_text', group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), n), random_state=random_state))  # Safe sampling
        .reset_index(drop=True)
    )
    
    # ETPR
    etpr_df = df[df['Q_TYPE'] == "Primary etiology"]
    etpr_group_sizes = etpr_df.groupby('ground_truth_text').size()
    etpr_max_n = min(etpr_group_sizes.max(), n_etpr)  # Don't oversample beyond n_etpr
    def etpr_oversample_group(x):
        n_to_add = etpr_max_n - len(x)
        if n_to_add <= 0:
            return x.sample(n=etpr_max_n, random_state=random_state)
        else:
            return pd.concat([x, x.sample(n=n_to_add, replace=True, random_state=random_state)], ignore_index=True)
    etpr_data = (
        etpr_df
        .groupby('ground_truth_text', group_keys=False)
        .apply(etpr_oversample_group)
        .reset_index(drop=True)
    )
    
    # Cogstat
    cog_data = (
        df[df['Q_TYPE'] == "Cognitive status"].apply(add_cog_type, axis=1)
        .groupby('COG_TYPE', group_keys=False)
        .apply(
            lambda cog_grp: cog_grp
                .groupby('ground_truth_text', group_keys=False)
                .apply(lambda x: x.sample(n=min(len(x), n), random_state=random_state))
                # .sample(n=min(len(cog_grp), 2 * n), random_state=random_state)
                .reset_index(drop=True)
        )
        .reset_index(drop=True)
    )
    cog_data.drop(['COG_TYPE'], inplace=True, axis=1)
    
    # cog_data = change_cog_cases(df[df['Q_TYPE'] == 'Cognitive status'], df[df['Q_TYPE'] == 'MCI subtype'], n=n)
    combined_data = pd.concat([np_data, datscan_data, amylpet_data, amylcsf_data, etpr_data, cog_data]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    # assert combined_data['ID'].is_unique, "combined_data has repeated IDs"
    
    return combined_data


In [ ]:
def options_str_to_dict(options_str):
    """
    Convert a string like 'A. Dementia (DE)\nB. Normal Cognition (NC)'
    to a dict: {'A': 'Dementia (DE)', 'B': 'Normal Cognition (NC)'}
    """
    option_dict = {}
    lines = options_str.split('\n')
    for line in lines:
        # Match lines that look like "A. Something"
        if '. ' in line:
            letter, option = line.split('. ', 1)
            letter = letter.strip()
            option = option.strip()
            option_dict[letter] = option
    return option_dict

def options_dict_to_str(options_dict):
    """
    Convert a dict like {'A': 'Dementia (DE)', 'B': 'Normal Cognition (NC)'}
    to a string: 'A. Dementia (DE)\nB. Normal Cognition (NC)'
    """
    lines = []
    for key in options_dict.keys():
        lines.append(f"{key}. {options_dict[key]}")
    return "\n".join(lines)

def shuffle_options_str(row):
    options_str = row['options']
    options_dict = options_str_to_dict(options_str)
    option_texts = list(options_dict.values())
    random.shuffle(option_texts)
    # Construct a new dict with letter keys starting from 'A'
    new_keys = [chr(ord('A') + i) for i in range(len(option_texts))]
    shuffled_dict = {k: v for k, v in zip(new_keys, option_texts)}
    # Find which new letter corresponds to the original ground_truth_text
    gt_text = row['ground_truth_text']
    shuffled_dict_v_k = {v:k for k,v in shuffled_dict.items()}
    # letter_for_gt = None
    # for k, v in shuffled_dict.items():
    #     if v == gt_text:
    #         letter_for_gt = k
    #         break
    row['ground_truth'] = shuffled_dict_v_k[gt_text]
    row['options'] = options_dict_to_str(shuffled_dict)
    return row

In [ ]:
def sample_cases_oversample_shuffle_options(df, n=500, n_etpr=300):
    # Neuropath
    np_df = df[df['Q_TYPE'] == "Neuropath"]
    group_sizes = np_df.groupby('ground_truth_text').size()
    max_n = group_sizes.max()
    def oversample_group(x):
        # Always include all original cases, then sample (with replacement) the remainder needed
        n_to_add = max_n - len(x)
        if n_to_add <= 0:
            return x
        else:
            remaining_samples = x.sample(n=n_to_add, replace=True, random_state=random_state)
            
            remaining_samples = remaining_samples.apply(shuffle_options_str, axis=1)
            return pd.concat([x, remaining_samples], ignore_index=True)
    np_data = (
        np_df
        .groupby('ground_truth_text', group_keys=False)
        .apply(oversample_group)
        .reset_index(drop=True)
    )
    
    # Datscan
    datscan_data = df[df['Q_TYPE'] == 'DATSCAN']
    
    # Amyloid
    amylpet_data = (
        df[df['Q_TYPE'] == "Amyloid PET"]
        .groupby('ground_truth_text', group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), n), random_state=random_state))  # Safe sampling
        .reset_index(drop=True)
    )
    amylcsf_data = (
        df[df['Q_TYPE'] == "Amyloid CSF"]
        .groupby('ground_truth_text', group_keys=False)
        .apply(lambda x: x.sample(n=min(len(x), n), random_state=random_state))  # Safe sampling
        .reset_index(drop=True)
    )
    
    # ETPR
    etpr_df = df[df['Q_TYPE'] == "Primary etiology"]
    etpr_group_sizes = etpr_df.groupby('ground_truth_text').size()
    etpr_max_n = min(etpr_group_sizes.max(), n_etpr)  # Don't oversample beyond n_etpr
    def etpr_oversample_group(x):
        n_to_add = etpr_max_n - len(x)
        if n_to_add <= 0:
            return x.sample(n=etpr_max_n, random_state=random_state)
        else:
            return pd.concat([x, x.sample(n=n_to_add, replace=True, random_state=random_state)], ignore_index=True)
    etpr_data = (
        etpr_df
        .groupby('ground_truth_text', group_keys=False)
        .apply(etpr_oversample_group)
        .reset_index(drop=True)
    )
    
    # Cogstat
    cog_data = (
        df[df['Q_TYPE'] == "Cognitive status"].apply(add_cog_type, axis=1)
        .groupby('COG_TYPE', group_keys=False)
        .apply(
            lambda cog_grp: cog_grp
                .groupby('ground_truth_text', group_keys=False)
                .apply(lambda x: x.sample(n=min(len(x), n), random_state=random_state))
                # .sample(n=min(len(cog_grp), 2 * n), random_state=random_state)
                .reset_index(drop=True)
        )
        .reset_index(drop=True)
    )
    cog_data.drop(['COG_TYPE'], inplace=True, axis=1)
    
    # cog_data = change_cog_cases(df[df['Q_TYPE'] == 'Cognitive status'], df[df['Q_TYPE'] == 'MCI subtype'], n=n)
    combined_data = pd.concat([np_data, datscan_data, amylpet_data, amylcsf_data, etpr_data, cog_data]).sample(frac=1, random_state=random_state).reset_index(drop=True)
    
    # assert combined_data['ID'].is_unique, "combined_data has repeated IDs"
    
    return combined_data


# Sample from all NACC cases

In [6]:
all_data_sample = sample_cases(all_data, n=500, n_etpr=300)

In [7]:
all_data['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    26722
Primary etiology    11342
Neuropath            2000
Amyloid PET          2000
Amyloid CSF           800
DATSCAN               200
Name: count, dtype: int64

In [8]:
all_data_sample['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    3000
Primary etiology    2088
Neuropath           2000
Amyloid CSF          500
Amyloid PET          500
DATSCAN              200
Name: count, dtype: int64

In [9]:
len(all_data_sample)

8288

In [10]:
for qtype in all_data_sample['Q_TYPE'].unique():
    print(f"Q_TYPE: {qtype}")
    print(all_data_sample[all_data_sample['Q_TYPE'] == qtype]['ground_truth_text'].value_counts())
    print()


Q_TYPE: Cognitive status
ground_truth_text
Normal Cognition (NC)              1000
Mild Cognitive Impairment (MCI)    1000
Dementia (DE)                      1000
Name: count, dtype: int64

Q_TYPE: Neuropath
ground_truth_text
Alzheimer's disease pathology (AD)                                                                                                                   633
Alzheimer's disease pathology (AD) and Lewy body pathology (LBD)                                                                                     585
Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)                                                                      299
No listed option is correct                                                                                                                          165
Lewy body pathology (LBD)                                                                                                                             97
Alzheimer

In [ ]:
all_data_sample.to_csv("../../data/nacc/training_data/training_data_grpo_modified_cog_np/subset/all_train_sub.csv", index=False)

# Sample from filtered NACC cases

In [7]:
filtered_data_sample = sample_cases(filtered_data, n=500, n_etpr=300)

In [8]:
filtered_data['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    17122
Primary etiology     8739
Amyloid PET          1635
Neuropath            1530
Amyloid CSF           660
DATSCAN               158
Name: count, dtype: int64

In [9]:
filtered_data_sample['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    3000
Primary etiology    1738
Neuropath           1530
Amyloid PET          500
Amyloid CSF          500
DATSCAN              158
Name: count, dtype: int64

In [10]:
len(filtered_data_sample)

7426

In [11]:
for qtype in filtered_data_sample['Q_TYPE'].unique():
    print(f"Q_TYPE: {qtype}")
    print(filtered_data_sample[filtered_data_sample['Q_TYPE'] == qtype]['ground_truth_text'].value_counts())
    print()


Q_TYPE: Neuropath
ground_truth_text
Alzheimer's disease pathology (AD) and Lewy body pathology (LBD)                                                                                     525
Alzheimer's disease pathology (AD)                                                                                                                   488
Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)                                                                      186
No listed option is correct                                                                                                                           92
Alzheimer's disease pathology (AD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)                                80
Lewy body pathology (LBD)                                                                                                                             74
Alzheimer's disease pathology (AD), Lewy body 

In [13]:
filtered_data_sample.to_csv(f"../../data/nacc/training_data/training_data_grpo_modified_cog_np/subset/{group_size}/train_filtered_sub.csv", index=False)

# Sample from stage wise NACC data

In [20]:
# stage_1_data_sample = sample_cases(stage_1_data, n=125, n_etpr=75)
# stage_2_data_sample = sample_cases(stage_2_data, n=125, n_etpr=75)
# stage_3_data_sample = sample_cases(stage_3_data, n=125, n_etpr=75)
# stage_4_data_sample = sample_cases(stage_4_data, n=125, n_etpr=75)
# stage_data_sample = pd.concat([stage_1_data_sample, stage_2_data_sample, stage_3_data_sample, stage_4_data_sample]).reset_index(drop=True)

In [ ]:
# stage_data = pd.concat([stage_1_data, stage_2_data, stage_3_data, stage_4_data]).reset_index(drop=True)

In [14]:
stage_data_sample = sample_cases(stage_wise_data, n=500, n_etpr=300)
stage_data_sample = stage_data_sample.sort_values(by='STAGE').reset_index(drop=True)


In [15]:
stage_data_sample.head()

,ID,patient_summary,diag_summary,question,options,ground_truth,ground_truth_text,Q_TYPE,visit_summary_prompt,visit_summary,COHORT,STAGE
0,NACC820318,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",Analyze the patient's information and identify...,A. Normal Cognition (NC)\nB. Dementia (DE),B,Dementia (DE),Cognitive status,<|im_start|>user\nYou will receive patient dat...,"The subject is a 77-year-old right-handed, Whi...",NACC,0
1,NACC522336,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",Does the patient likely have abnormal amyloid ...,A. No\nB. Yes,A,No,Amyloid PET,<|im_start|>user\nYou will receive patient dat...,The subject is a 67-year-old male who identifi...,NACC,0
2,NACC622640,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",Is the patient likely to have elevated amyloid...,A. Yes\nB. No,B,No,Amyloid PET,<|im_start|>user\nYou will receive patient dat...,The subject is a 43-year-old male of Asian des...,NACC,0
3,NACC794444,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",Identify the primary etiology of the patient's...,A. Not applicable (no cognitive impairment)\nB...,A,Not applicable (no cognitive impairment),Primary etiology,<|im_start|>user\nYou will receive patient dat...,The subject is a 68-year-old White female who ...,NACC,0
4,NACC774395,"{""Subject Demographics"": {""Subject's age at vi...","{""Clinician Judgment of Symptoms"": {""Does the ...",Analyze the patient's information and identify...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,B,Normal Cognition (NC),Cognitive status,<|im_start|>user\nYou will receive patient dat...,"The subject is a 76-year-old right-handed, Whi...",NACC,0


In [16]:
stage_wise_data['STAGE'].value_counts()

STAGE
0    13841
3     6616
1     6379
2     5235
Name: count, dtype: int64

In [17]:
stage_data_sample['STAGE'].value_counts()

STAGE
0    2936
3    2274
1    1505
2    1318
Name: count, dtype: int64

In [18]:
stage_wise_data['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    17499
Primary etiology    10241
Neuropath            1781
Amyloid PET          1704
Amyloid CSF           682
DATSCAN               164
Name: count, dtype: int64

In [19]:
stage_data_sample['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    3000
Primary etiology    2088
Neuropath           1781
Amyloid PET          500
Amyloid CSF          500
DATSCAN              164
Name: count, dtype: int64

In [20]:
len(stage_data_sample)

8033

In [23]:
for qtype in stage_data_sample['Q_TYPE'].unique():
    print(f"Q_TYPE: {qtype}")
    print(stage_data_sample[stage_data_sample['Q_TYPE'] == qtype]['ground_truth_text'].value_counts())
    print()


Q_TYPE: Cognitive status
ground_truth_text
Dementia (DE)                      1000
Normal Cognition (NC)              1000
Mild Cognitive Impairment (MCI)    1000
Name: count, dtype: int64

Q_TYPE: Amyloid PET
ground_truth_text
No     250
Yes    250
Name: count, dtype: int64

Q_TYPE: Primary etiology
ground_truth_text
Not applicable (no cognitive impairment)                                                                                                                                                                             300
Alzheimer's disease (AD)                                                                                                                                                                                             300
Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)    300
Vascular brain injury or vasc

In [24]:
stage_data_sample.to_csv(f"../../data/nacc/training_data/training_data_grpo_modified_cog_np/subset/{group_size}/stage_wise_sub.csv", index=False)

# Increased Sample from all NACC cases

In [12]:
all_data_inc_sample = sample_cases(all_data, n=1000, n_etpr=1000)

In [13]:
all_data['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    26722
Primary etiology    11342
Neuropath            2000
Amyloid PET          2000
Amyloid CSF           800
DATSCAN               200
Name: count, dtype: int64

In [14]:
all_data_inc_sample['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    6000
Primary etiology    5047
Amyloid PET         2000
Neuropath           2000
Amyloid CSF          800
DATSCAN              200
Name: count, dtype: int64

In [15]:
len(all_data_inc_sample)

16047

In [16]:
for qtype in all_data_inc_sample['Q_TYPE'].unique():
    print(f"Q_TYPE: {qtype}")
    print(all_data_inc_sample[all_data_inc_sample['Q_TYPE'] == qtype]['ground_truth_text'].value_counts())
    print()


Q_TYPE: Cognitive status
ground_truth_text
Mild Cognitive Impairment (MCI)    2000
Dementia (DE)                      2000
Normal Cognition (NC)              2000
Name: count, dtype: int64

Q_TYPE: Primary etiology
ground_truth_text
Not applicable (no cognitive impairment)                                                                                                                                                                             1000
Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)    1000
Alzheimer's disease (AD)                                                                                                                                                                                             1000
Lewy body disease (LBD)                                                                                          

In [ ]:
all_data_inc_sample.to_csv("../../data/training_data/training_data_grpo_modified_cog_np_HASH/subset/all_train_inc_sub.csv", index=False)

# Increased Sample with oversampling from all NACC cases

In [17]:
all_train_inc_oversample = sample_cases_oversample(all_data, n=1300, n_etpr=1000)

In [18]:
duplicate_ids = all_train_inc_oversample['ID'].value_counts()
rows_with_duplicates = all_train_inc_oversample[all_train_inc_oversample['ID'].isin(duplicate_ids[duplicate_ids > 1].index)]

In [19]:
rows_with_duplicates['ground_truth_text'].value_counts()

ground_truth_text
Other (Multiple system atrophy, Essential tremor, Down syndrome, Huntington's disease, Prion disease, Traumatic brain injury, Normal-pressure hydrocephalus, Epilepsy, CNS neoplasm, etc)    1000
Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)                                                                 998
Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)                       958
Vascular brain injury or vascular dementia including stroke (VD)                                                                                                                              709
Lewy body pathology (LBD)                                                                                                                                                                     633
Alzheimer's 

In [20]:
len(rows_with_duplicates[rows_with_duplicates['ground_truth_text'] == "Lewy body pathology (LBD)"]['ID'].value_counts())

97

In [21]:
for qtype in all_data['Q_TYPE'].unique():
    print(f"Q_TYPE: {qtype}")
    print(all_data[all_data['Q_TYPE'] == qtype]['ground_truth_text'].value_counts())
    print()


Q_TYPE: Cognitive status
ground_truth_text
Normal Cognition (NC)              13468
Dementia (DE)                       7283
Mild Cognitive Impairment (MCI)     5971
Name: count, dtype: int64

Q_TYPE: Neuropath
ground_truth_text
Alzheimer's disease pathology (AD)                                                                                                                   633
Alzheimer's disease pathology (AD) and Lewy body pathology (LBD)                                                                                     585
Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)                                                                      299
No listed option is correct                                                                                                                          165
Lewy body pathology (LBD)                                                                                                                             97
Alzhei

In [22]:
all_data['Q_TYPE'].value_counts()

Q_TYPE
Cognitive status    26722
Primary etiology    11342
Neuropath            2000
Amyloid PET          2000
Amyloid CSF           800
DATSCAN               200
Name: count, dtype: int64

In [23]:
all_train_inc_oversample['Q_TYPE'].value_counts()

Q_TYPE
Primary etiology    8000
Cognitive status    7800
Neuropath           5064
Amyloid PET         2000
Amyloid CSF          800
DATSCAN              200
Name: count, dtype: int64

In [24]:
len(all_train_inc_oversample)

23864

In [25]:
len(all_train_inc_oversample['ID'].unique())

17847

In [26]:
for qtype in all_train_inc_oversample['Q_TYPE'].unique():
    print(f"Q_TYPE: {qtype}")
    print(all_train_inc_oversample[all_train_inc_oversample['Q_TYPE'] == qtype]['ground_truth_text'].value_counts())
    print()


Q_TYPE: Neuropath
ground_truth_text
Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)                                        633
Alzheimer's disease pathology (AD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)                               633
Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)                                                                      633
Alzheimer's disease pathology (AD), Lewy body pathology (LBD) and Frontotemporal Lobar Degeneration with tau pathology or TDP-43 pathology (FTLD)    633
Lewy body pathology (LBD)                                                                                                                            633
Alzheimer's disease pathology (AD)                                                                                                                   633
Alzheimer's disease pathology (AD) and Lewy bo

In [ ]:
all_train_inc_oversample.to_csv("../../data/training_data/training_data_grpo_modified_cog_np_HASH/subset/all_train_inc_oversample_sub.csv", index=False)

# Increased Sample with oversampling from all NACC cases with option shuffling

In [ ]:
df = pd.read_csv("../../data/training_data/subset/all_train_inc_oversample_sub.csv")

In [ ]:
all_train_inc_oversample_shuffle = sample_cases_oversample_shuffle_options(all_data, n=1300, n_etpr=1000)

In [ ]:
len(set(all_train_inc_oversample_shuffle['ID']))

In [ ]:
len(set(all_train_inc_oversample_shuffle['ID']).intersection(set(df['ID'])))

In [ ]:
duplicate_ids = all_train_inc_oversample_shuffle['ID'].value_counts()
rows_with_duplicates = all_train_inc_oversample_shuffle[all_train_inc_oversample_shuffle['ID'].isin(duplicate_ids[duplicate_ids > 1].index)]

In [ ]:
rows_with_duplicates.head()

In [ ]:
rows_with_duplicates['ground_truth_text'].value_counts()

In [ ]:
for qtype in all_data['Q_TYPE'].unique():
    print(f"Q_TYPE: {qtype}")
    print(all_data[all_data['Q_TYPE'] == qtype]['ground_truth_text'].value_counts())
    print()


In [ ]:
all_data['Q_TYPE'].value_counts()

In [ ]:
all_train_inc_oversample_shuffle['Q_TYPE'].value_counts()

In [ ]:
len(all_train_inc_oversample_shuffle)

In [ ]:
len(all_train_inc_oversample_shuffle['ID'].unique())

In [ ]:
for qtype in all_train_inc_oversample_shuffle['Q_TYPE'].unique():
    print(f"Q_TYPE: {qtype}")
    print(all_train_inc_oversample_shuffle[all_train_inc_oversample_shuffle['Q_TYPE'] == qtype]['ground_truth_text'].value_counts())
    print()


In [ ]:
all_train_inc_oversample_shuffle.to_csv("../../data/training_data/training_data_grpo_modified_cog_np_HASH/subset/all_train_inc_oversample_shuffle_sub.csv", index=False)